# BraTS2020 UNet - colab version

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")


attach google drive for checkpointing:

In [ ]:
USE_DRIVE = True

import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/brats2020'
else:
    PROJECT_DIR = '/content/brats2020'

os.makedirs(PROJECT_DIR, exist_ok=True)

DATA_DIR = '/content/data'
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Project dir (checkpoints):", PROJECT_DIR)
print("Data dir:", DATA_DIR)

download dataset from kaggle:

In [ ]:
import glob, os

existing = glob.glob(os.path.join(DATA_DIR, '**', '*.h5'), recursive=True)

if existing:
    print(f"Found {len(existing)} existing .h5 files in {DATA_DIR}")
else:
    from getpass import getpass
    kaggleToken = getpass("Enter kaggle API token: ")

    os.makedirs('/root/.kaggle', exist_ok=True)

    os.environ['KAGGLE_API_TOKEN'] = kaggleToken
    with open('/root/.kaggle/access_token', 'w') as f:
        f.write(kaggleToken)
    os.chmod('/root/.kaggle/access_token', 0o600)

    %pip install -q -U kaggle
    %kaggle datasets download -d awsaf49/brats2020-training-data -p "{DATA_DIR}" --unzip

In [ ]:
#find where in directory .h5 files are located
h5Files = glob.glob(os.path.join(DATA_DIR, '**', '*.h5'), recursive=True)
if not h5Files:
    raise FileNotFoundError(f"No .h5 files found in {DATA_DIR}")

DATA_PATH = os.path.dirname(h5Files[0])
matching = glob.glob(os.path.join(DATA_PATH, '*.h5'))
print("DATA_PATH set to", DATA_PATH)
print(len(matching), ".h5 files found")


define unet (model.py):

In [5]:
import torch
from torch import nn

class DoubleConvBlock(nn.Module):
    def __init__(self, inputDepth, outputDepth):
        super().__init__()

        kernelSize = 3
        stride = 1
        padding = 1

        self.doubleConv = nn.Sequential(
            nn.Conv2d(inputDepth, outputDepth, kernelSize, stride, padding, bias=False),
            nn.BatchNorm2d(outputDepth),
            nn.ReLU(inplace = True),
            nn.Conv2d(outputDepth, outputDepth, kernelSize, stride, padding, bias=False),
            nn.BatchNorm2d(outputDepth),
            nn.ReLU(inplace = True)
        )

    def forward(self, xIn):
        return self.doubleConv(xIn)

class DownBlock(nn.Module):
    def __init__(self, inputDepth, outputDepth):
        super().__init__()

        self.down = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConvBlock(inputDepth, outputDepth)
        )

    def forward(self, xIn):
        return self.down(xIn)

class UpBlock(nn.Module):
    def __init__(self, inputDepth, outputDepth):
        super().__init__()

        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False), #doubles width and height of tensor
            nn.Conv2d(inputDepth, inputDepth // 2, 1) #need to half depth
        )

        self.conv = DoubleConvBlock(inputDepth, outputDepth)

    def forward(self, xIn, xBridged):
        xIn = self.up(xIn)
        return self.conv(torch.concatenate([xIn, xBridged], dim=1))

class UNet(nn.Module):
    def __init__(self, layerDepths): #layerdepths is the depth/channels in each layer tensor
        super().__init__()

        outputDepth = layerDepths[-1]
        layerDepths = layerDepths[:-1]

        self.inConv = DoubleConvBlock(layerDepths[0], layerDepths[1])

        self.downBlocks = nn.ModuleList()
        for i in range(len(layerDepths[:-2])):
            self.downBlocks.append(DownBlock(layerDepths[i+1], layerDepths[i+2]))

        self.upBlocks = nn.ModuleList()
        upDepths = layerDepths[1:][::-1]

        for i in range(len(upDepths[:-1])):
            self.upBlocks.append(UpBlock(upDepths[i], upDepths[i+1]))

        self.outConv = nn.Conv2d(upDepths[-1], outputDepth, kernel_size=1)

    def forward(self, x):
        bridgedTensors = []

        x = self.inConv(x)
        bridgedTensors.append(x)

        for downBlock in self.downBlocks:
            x = downBlock(x)
            bridgedTensors.append(x)

        bridgedTensors.pop()

        for upBlock in self.upBlocks:
            xBridged = bridgedTensors.pop()
            x = upBlock(x, xBridged)

        return self.outConv(x)


dataset and cost function:

In [8]:
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split

class Data(Dataset):
    def __init__(self, dataPath):
        self.filePaths = glob.glob(dataPath + "/*.h5")

    def __len__(self):
        return len(self.filePaths)

    def __getitem__(self, i):
        with h5py.File(self.filePaths[i], "r") as file:
            image = file["image"][:].astype(np.float32)
            mask = file["mask"][:].astype(np.float32)

        inputTensor = torch.from_numpy(image.transpose(2, 0, 1))
        targets = torch.from_numpy(mask.transpose(2, 0, 1))

        return inputTensor, targets

def costFunction(inputTensor, targets):
    binaryCrossEntropy = torch.nn.functional.binary_cross_entropy_with_logits(inputTensor, targets)

    sigmoidInputs = torch.sigmoid(inputTensor)

    flattenedInputs = sigmoidInputs.view(sigmoidInputs.size(0), sigmoidInputs.size(1), -1)
    flattenedTargets = targets.view(targets.size(0), targets.size(1), -1)

    totalCorrect = (flattenedInputs * flattenedTargets).sum(dim=2)
    smoothing = 0.0001
    diceScore = (2 * totalCorrect + smoothing) / (flattenedInputs.sum(dim=2) + flattenedTargets.sum(dim=2) + smoothing)
    dice = (1 - diceScore).mean()

    return (binaryCrossEntropy + dice) / 2


training setup

In [ ]:
import torch.optim as optimisers

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

EPOCHS = 25
LEARNING_RATE = 0.001
BATCH_SIZE = 6
NUM_WORKERS = 2  # set to 0 if colab gives DataLoader worker errors

data = Data(DATA_PATH)
print("Total samples found:", len(data))

trainingSplit = int(0.8 * len(data))
trainingData, testingData = random_split(
    data, [trainingSplit, len(data) - trainingSplit], generator=torch.Generator().manual_seed(19)
)

trainingDataLoader = DataLoader(trainingData, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
testingDataLoader = DataLoader(testingData, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

model = UNet([4, 16, 32, 64, 128, 3]).to(device)
optimiser = optimisers.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=0.00001)
lrScheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, patience=6, factor=0.5, min_lr=0.000001)

def trainEpoch(model, optimiser, dataLoader):
    model.train()
    totalCost = 0
    for inputTensor, targets in dataLoader:
        inputTensor, targets = inputTensor.to(device), targets.to(device)
        optimiser.zero_grad()
        cost = costFunction(model(inputTensor), targets)
        cost.backward()
        optimiser.step()
        totalCost += cost.item() * inputTensor.size(0)

    return totalCost / len(dataLoader.dataset)

@torch.no_grad()
def test(model, dataLoader):
    model.eval()
    totalCost = 0
    for inputTensor, targets in dataLoader:
        inputTensor, targets = inputTensor.to(device), targets.to(device)
        cost = costFunction(model(inputTensor), targets)
        totalCost += cost.item() * inputTensor.size(0)

    return totalCost / len(dataLoader.dataset)


if checkpoint exists, resume from checkpoint:

In [ ]:
checkpointPath = os.path.join(CHECKPOINT_DIR, "checkpoint.pth")

startEpoch = 0
lowestCost = float("inf")
trainingEvolution = []
testingEvolution = []

if os.path.exists(checkpointPath):
    checkpoint = torch.load(checkpointPath, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    optimiser.load_state_dict(checkpoint["optimiser_state"])
    lrScheduler.load_state_dict(checkpoint["scheduler_state"])
    startEpoch = checkpoint["epoch"] + 1
    lowestCost = checkpoint["lowestCost"]
    trainingEvolution = checkpoint["trainingEvolution"]
    testingEvolution = checkpoint["testingEvolution"]
    print(f"continuing from epoch {startEpoch} (lowest cost so far: {lowestCost:.4f})")
else:
    print("no checkpoint found")


train model:
 - if runtime disconnects, rerun ^ checkpoint cell then start training again

In [ ]:
print("training model...")

for i in range(startEpoch, EPOCHS):
    trainingCost = trainEpoch(model, optimiser, trainingDataLoader)
    testingCost = test(model, testingDataLoader)
    lrScheduler.step(testingCost)

    print("Epoch", i, ":\ntraining cost =", trainingCost, "\ntesting cost =", testingCost)

    trainingEvolution.append(trainingCost)
    testingEvolution.append(testingCost)

    isBest = testingCost < lowestCost
    if isBest:
        lowestCost = testingCost
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "bestModel.pth"))

    #save a checkpoint every epoch
    torch.save({
        "epoch": i,
        "model_state": model.state_dict(),
        "optimiser_state": optimiser.state_dict(),
        "scheduler_state": lrScheduler.state_dict(),
        "lowestCost": lowestCost,
        "trainingEvolution": trainingEvolution,
        "testingEvolution": testingEvolution,
    }, os.path.join(CHECKPOINT_DIR, "checkpoint.pth"))

print("\nFinal training cost =", trainingCost, "\nFinal testing cost =", testingCost)

torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "model.pth"))
print("model saved to", os.path.join(CHECKPOINT_DIR, "model.pth"))
print("best model saved to", os.path.join(CHECKPOINT_DIR, "bestModel.pth"), "- testing cost:", lowestCost)


evaluate best model average dice score:

In [ ]:
@torch.no_grad()
def evaluateDice(model, dataLoader):
    model.eval()
    totalDice = 0
    totalSamples = 0

    for inputTensor, targets in dataLoader:
        inputTensor, targets = inputTensor.to(device), targets.to(device)

        sigmoidInputs = torch.sigmoid(model(inputTensor))

        flattenedInputs = sigmoidInputs.view(sigmoidInputs.size(0), sigmoidInputs.size(1), -1)
        flattenedTargets = targets.view(targets.size(0), targets.size(1), -1)

        totalCorrect = (flattenedInputs * flattenedTargets).sum(dim=2)
        smoothing = 0.0001
        diceScore = (2 * totalCorrect + smoothing) / (
            flattenedInputs.sum(dim=2) + flattenedTargets.sum(dim=2) + smoothing
        )

        perSampleDice = diceScore.mean(dim=1)
        totalDice += perSampleDice.sum().item()
        totalSamples += inputTensor.size(0)

    return totalDice / totalSamples

#load best checkpoint so this cell can be run whenever
model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "bestModel.pth"), map_location=device))

finalDiceScore = evaluateDice(model, testingDataLoader)
print(f"Average dice score on test set: {finalDiceScore:.4f}")
print(f"equivalent to {finalDiceScore * 100:.2f}%")

visualise training:

In [ ]:
import matplotlib.pyplot as plt

plt.plot(trainingEvolution, label="training cost")
plt.plot(testingEvolution, label="testing cost")
plt.xlabel("Epoch")
plt.ylabel("Cost")
plt.legend()
plt.show()


download trained model if drive is not attached

In [ ]:
if not USE_DRIVE:
    from google.colab import files
    files.download(os.path.join(CHECKPOINT_DIR, "bestModel.pth"))
